Append a batch of new click-stream events to raw.order_events.

In [1]:
import sys, random
from datetime import datetime, timedelta
import duckdb

In [2]:
n = 500
DB = "shopflow.duckdb"

In [3]:
# Parameters
n = 500
DB = "shopflow_prod.duckdb"


In [4]:
con = duckdb.connect(DB)

In [5]:
con.execute("""
            create table if not exists raw.order_events(
                event_id bigint, order_id bigint, event_type varchar, event_at timestamp
            )
            """)

In [6]:
last_id, last_at = con.execute("""
                            select 
                                coalesce(max(event_id), 0),
                                coalesce(max(event_at), timestamp '2026-11-01 00:00:00')
                            from raw.order_events
                               """).fetchone()

In [7]:
rows = []
for i in range(1, n + 1):
    rows.append((last_id + 1, random.randint(1, 1200),
                 random.choice(["viewed", "added_to_cart", "checked_out"]),
                 last_at + timedelta(seconds=i)
                 ))
    
con.executemany("insert into raw.order_events values (?,?,?,?)", rows)
total = con.execute("select count(*) from raw.order_events").fetchone()[0]

In [8]:
print(f"added {n} events, raw.order_events new has {total} rows")

added 500 events, raw.order_events new has 500 rows


In [9]:
con.close()

In [10]:
# Run this
# > papermill new_events.ipynb output_new_events.ipynb -p n 500 -p DB "shopflow_prod.duckdb"